In [1]:
import string
import re
import pandas as pd
import spacy
import inflect
from sentence_transformers import SentenceTransformer, util
from rank_bm25 import BM25Okapi
import numpy as np
from scipy.stats import spearmanr
from nltk.stem import PorterStemmer

stemmer = PorterStemmer()
p = inflect.engine()
nlp = spacy.load("en_core_sci_lg")
model = SentenceTransformer("all-MiniLM-L6-v2")

/Users/alisa/anaconda3/lib/python3.11/site-packages/spacy/language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]


In [67]:
QSTN = "What is the recommended method for measuring blood pressure in clinical practice according to the text?".lower()
REFANS = "The text recommends attended office BP measurement using an automated device and a standardized protocol (triplicate measurements in appropriate conditions and position) as the most reasonable and practical BP measurement approach for clinical practice..".lower() #good - reference
ANS2 = "The recommended method for measuring blood pressure in clinical practice according to the text is using an automated device and a standardized protocol with triplicate measurements in appropriate conditions and position.".lower() #good
ANS1 = "To ensure accurate blood pressure (BP) readings in clinical settings, the text advocates utilizing an automated device in conjunction with a standardized protocol. This protocol entails taking triplicate measurements under appropriate conditions and positions.".lower() #wrong
STDANS = "The text recommends measuring blood pressure by shouting at the patient loudly."

In [68]:
def replace_separators_with_space(input_string):
    separators = ["-", "_", ".", ",", ";", ":", "/", "\\", "|", "!", "?", "'"]
    for separator in separators:
        input_string = input_string.replace(separator, " ")
    return input_string

def remove_question_entities(question, answer):
    Q_processed = nlp(question)
    ANS_processed = nlp(answer)
    Q_entities = set([word for ent in list(Q_processed.ents) for word in ent.text.split()])
    #ANS_entities = set([word for ent in list(ANS_processed.ents) for word in ent.text.split()])
    #ANS_toremove = ANS_entities.intersection(Q_entities) #meh is it really needed tho?
    ANS_toremove = Q_entities
    clean_ans = answer_cleaner(answer, ANS_toremove)
    return clean_ans

def answer_cleaner(orig, setRem):
    orig = orig.lower()
    for wrd in setRem:
        orig = orig.replace(wrd,'')
    return orig

def string_cleaner(sentence):
    punctuations = string.punctuation
    stop_words = nlp.Defaults.stop_words
    sentence = re.sub(r'\d+', r' \g<0> ', sentence)
    sentence = replace_separators_with_space(sentence)
    tokens = nlp(sentence)
    tokens = [word.lemma_.lower().strip() for word in tokens]
    tokens = [word for word in tokens if (word not in stop_words and word not in punctuations) or word.isdigit()]
    tokens = [word if not word.isdigit() else p.number_to_words(word) for word in tokens]
    sentence = ' '.join(tokens)
    sentence = sentence.replace('<',' less than ')
    sentence = sentence.replace('>',' more than ')
    sentence = sentence.replace('≥',' greater than or equal to ')
    sentence = sentence.replace('≤',' less than or equal to ')
    sentence = sentence.replace('=',' equal to ')
    return sentence

In [69]:
def cosineSim(reference, answer):
    """
    :param reference:
    :param answer:
    :return:
    cosine similarity between the BERT encoded strings
    """
    REFANS_embedding = model.encode(reference)
    STDANS_embedding = model.encode(answer)
    return str(util.cos_sim(REFANS_embedding, STDANS_embedding).item())

def sort_by_gap(t):
    squaredError1 = (100 - t[1]*100)**2 + (t[2]*100)**2
    return squaredError1

def compare_entities(reference, std_answer):
    """
    :param reference:
    :param std_answer:
    :return:
    splitted lemma matching
    """
    ref_processed = nlp(reference)
    std_processed = nlp(std_answer)
    ref_ents = [stemmer.stem(token) for ent in ref_processed.ents for token in ent.text.lower().split()]
    std_ents = [stemmer.stem(token) for ent in std_processed.ents for token in ent.text.lower().split()]
    ref_NUM = [token.text for token in ref_processed if token.pos_ == "NUM"]
    std_NUM = [token.text for token in std_processed if token.pos_ == "NUM"]
    ref_keywords = set(ref_ents + ref_NUM)
    std_keywords = set(std_ents + std_NUM)
    intersection = ref_keywords.intersection(std_keywords) #number of tokens that are in both answers
    if len(ref_keywords) == 0 or len(std_keywords) == 0 or len(intersection) ==0:
        return '0'
    precision = (len(intersection)/len(std_keywords))
    recall = len(intersection)/len(ref_keywords)
    return str((2*precision*recall) / (precision + recall))
    #return str(len(intersection)/len(ref_keywords))

def compare_bm25_binary(reference_ranking, std_answer_ranking, precentile=80):
    """
    :param reference_ranking:
    :param std_answer_ranking:
    :param precentile:
    :return:
    Matching indexes precentage
    """
    ref_precentile = np.percentile(reference_ranking,precentile)
    std_precentile = np.percentile(std_answer_ranking, precentile)
    reference_ranking = [1 if x >= ref_precentile else 0 for x in reference_ranking]
    std_answer_ranking =  [1 if x >= std_precentile else 0 for x in std_answer_ranking]
    binary_score_list = [1 if a == b == 1 else 0 for a, b in zip(reference_ranking, std_answer_ranking)]
    binary_score = sum(binary_score_list)/sum(reference_ranking)
    return str(binary_score)

def compare_bm25(reference_ranking, std_answer_ranking, precentile=80):
    """
    :param reference_ranking:
    :param std_answer_ranking:
    :param precentile:
    :return:
    Spearman Coefficient
    """
    ref_precentile = np.percentile(reference_ranking,precentile)
    std_precentile = np.percentile(std_answer_ranking, precentile)
    reference_ranking = [x if x >= ref_precentile else 0 for x in reference_ranking]
    std_answer_ranking =  [x if x >= std_precentile else 0 for x in std_answer_ranking]
    spearman_corr , p_value = spearmanr(reference_ranking,std_answer_ranking)
    return str(spearman_corr)



In [70]:
doc = nlp(REFANS)
docs = nlp(STDANS)
doc.similarity(docs)

0.6705130319672106

In [71]:
rc = string_cleaner(REFANS)
sc = string_cleaner(STDANS)
rcen = remove_question_entities(string_cleaner(QSTN), rc)
scen = remove_question_entities(string_cleaner(QSTN), sc)
with open('output/chunks/19_chunk_2.txt') as file:
    chunk = file.read()
chunk = nlp(chunk)
sents = [x.text.split() for x in chunk.sents]
bm25 = BM25Okapi(sents)
r_scores = bm25.get_scores(REFANS.split())
c_scores = bm25.get_scores(STDANS.split())
rc_scores = bm25.get_scores(rc.split())
sc_scores = bm25.get_scores(sc.split())
rcen_scores = bm25.get_scores(rcen.split())
scen_scores = bm25.get_scores(scen.split())

print("Entity Comparison:")
print("(original):\t\t",compare_entities(REFANS,STDANS))
print("(clean):\t\t",compare_entities(rc,sc))
print("(NO Qents):\t\t",compare_entities(rcen,scen))
print("\nCosine Similarity")
print("(original):\t\t",cosineSim(REFANS,STDANS))
print("(clean):\t\t",cosineSim(rc,sc))
print("(NO Qents):\t\t",cosineSim(rcen,scen))
print("\nBM25 Binary")
print("(original):\t\t",compare_bm25_binary(r_scores,c_scores,80))
print("(clean):\t\t",compare_bm25_binary(rc_scores,sc_scores,80))
print("(NO Qents):\t\t",compare_bm25_binary(rcen_scores,scen_scores,80))
print("\nBM25")
print("(original):\t\t",compare_bm25(r_scores,c_scores,80))
print("(clean):\t\t",compare_bm25(rc_scores,sc_scores,80))
print("(NO Qents):\t\t",compare_bm25(rcen_scores,scen_scores,80))

Entity Comparison:
(original):		 0.1
(clean):		 0.2857142857142857
(NO Qents):		 0.1818181818181818

Cosine Similarity
(original):		 0.43688327074050903
(clean):		 0.559536874294281
(NO Qents):		 0.35789036750793457

BM25 Binary
(original):		 0.13333333333333333
(clean):		 1.0
(NO Qents):		 1.0

BM25
(original):		 -0.07605654328184203
(clean):		 -0.06537347277517416
(NO Qents):		 -0.023723093062946062


In [7]:
REFANS = " Neurological examination and cognitive status • Fundoscopic examination for hypertensive retinopathy in emergencies • Auscultation of heart and carotid arteries • Palpation of carotid and peripheral arteries • Ankle–brachial index" #good - reference
STDANS = "Neurological examination and cognitive status • Fundoscopic examination for hypertensive retinopathy in emergencies •"

In [8]:
z = pd.read_csv('full_25.csv', delimiter='\t', index_col=0)

z['prf_score'] = 'N/A'
z['prf_clean_score'] = 'N/A'
z['prf_noent_score'] = 'N/A'

z['good_score'] = 'N/A'
z['good_clean_score'] = 'N/A'
z['good_noent_score'] = 'N/A'

z['fair_score'] = 'N/A'
z['fair_clean_score'] = 'N/A'
z['fair_noent_score'] = 'N/A'

z['partial_score'] = 'N/A'
z['partial_clean_score'] = 'N/A'
z['partial_noent_score'] = 'N/A'

z['wrong_score'] = 'N/A'
z['wrong_clean_score'] = 'N/A'
z['wrong_noent_score'] = 'N/A'

newData = z[['chunk','question','reference_answer','perfect_answer','prf_score','prf_clean_score','prf_noent_score','good_answer','good_score','good_clean_score','good_noent_score','fair_answer','fair_score','fair_clean_score','fair_noent_score','partially_incorrect_answer','partial_score','partial_clean_score','partial_noent_score','wrong_answer','wrong_score','wrong_clean_score','wrong_noent_score']]

In [9]:
org_prf_list_entities,org_prf_list_similarities,org_prf_list_BM25B,org_prf_list_BM25=[],[],[],[]
clean_prf_list_entities,clean_prf_list_similarities,clean_prf_list_BM25B,clean_prf_list_BM25=[],[],[],[]
noent_prf_list_entities,noent_prf_list_similarities,noent_prf_list_BM25B,noent_prf_list_BM25=[],[],[],[]
org_good_list_entities,org_good_list_similarities,org_good_list_BM25B,org_good_list_BM25=[],[],[],[]
clean_good_list_entities,clean_good_list_similarities,clean_good_list_BM25B,clean_good_list_BM25=[],[],[],[]
noent_good_list_entities,noent_good_list_similarities,noent_good_list_BM25B,noent_good_list_BM25=[],[],[],[]
org_wrong_list_entities,org_wrong_list_similarities,org_wrong_list_BM25B,org_wrong_list_BM25=[],[],[],[]
clean_wrong_list_entities,clean_wrong_list_similarities,clean_wrong_list_BM25B,clean_wrong_list_BM25=[],[],[],[]
noent_wrong_list_entities,noent_wrong_list_similarities,noent_wrong_list_BM25B,noent_wrong_list_BM25=[],[],[],[]

In [10]:
for index, row in newData.iterrows():
    chunk_num = row['chunk']
    print(chunk_num)
    with open(f'output/chunks/{chunk_num}') as file:
        chunk_text = file.read()
    QSTN = row['question']
    clean_QSTN = string_cleaner(QSTN)
    ref_ans = row['perfect_answer']
    clean_ref = string_cleaner(ref_ans)
    noent_ref = remove_question_entities(clean_QSTN, clean_ref)
    prf_ans = row['reference_answer']
    clean_prf = string_cleaner(prf_ans)
    noent_prf = remove_question_entities(clean_QSTN, clean_prf)
    good_ans = row['good_answer']
    clean_good = string_cleaner(good_ans)
    noent_good = remove_question_entities(clean_QSTN, clean_good)
    fair_ans = row['fair_answer']
    clean_fair = string_cleaner(fair_ans)
    noent_fair = remove_question_entities(clean_QSTN, clean_fair)
    partial_ans = row['partially_incorrect_answer']
    clean_partial = string_cleaner(partial_ans)
    noent_partial = remove_question_entities(clean_QSTN, clean_partial)
    wrong_ans = row['wrong_answer']
    clean_wrong = string_cleaner(wrong_ans)
    noent_wrong = remove_question_entities(clean_QSTN, clean_wrong)

    chunk = nlp(chunk)
    sents = [x.text.split() for x in chunk.sents]
    bm25 = BM25Okapi(sents)

    ref_score = bm25.get_scores(ref_ans.split())
    clean_ref_score = bm25.get_scores(clean_ref.split())
    noent_ref_score = bm25.get_scores(noent_ref.split())

    prf_score = bm25.get_scores(prf_ans.split())
    clean_prf_score = bm25.get_scores(clean_prf.split())
    noent_prf_score = bm25.get_scores(noent_prf.split())

    good_score = bm25.get_scores(good_ans.split())
    clean_good_score = bm25.get_scores(clean_good.split())
    noent_good_score = bm25.get_scores(noent_good.split())

    fair_score = bm25.get_scores(fair_ans.split())
    clean_fair_score = bm25.get_scores(clean_fair.split())
    noent_fair_score = bm25.get_scores(noent_fair.split())

    partial_score = bm25.get_scores(partial_ans.split())
    clean_partial_score = bm25.get_scores(clean_partial.split())
    noent_partial_score = bm25.get_scores(noent_partial.split())

    wrong_score = bm25.get_scores(wrong_ans.split())
    clean_wrong_score = bm25.get_scores(clean_wrong.split())
    noent_wrong_score = bm25.get_scores(noent_wrong.split())

    newData.at[index,'prf_score'] = f"Entities: {compare_entities(ref_ans,prf_ans)}\nSimilarity: {cosineSim(ref_ans,prf_ans)}\nBM25B: {compare_bm25_binary(ref_score,prf_score,80)}\nBM25: {compare_bm25(ref_score,prf_score,80)}"
    newData.at[index,'prf_clean_score'] = f"Entities: {compare_entities(clean_ref,clean_prf)}\nSimilarity: {cosineSim(clean_ref,clean_prf)}\nBM25B: {compare_bm25_binary(clean_ref_score,clean_prf_score,80)}\nBM25: {compare_bm25(clean_ref_score,clean_prf_score,80)}"
    newData.at[index,'prf_noent_score'] = f"Entities: {compare_entities(noent_ref,noent_prf)}\nSimilarity: {cosineSim(noent_ref,noent_prf)}\nBM25B: {compare_bm25_binary(noent_ref_score,noent_prf_score,80)}\nBM25: {compare_bm25(noent_ref_score,noent_prf_score,80)}"

    newData.at[index,'good_score'] = f"Entities: {compare_entities(ref_ans,good_ans)}\nSimilarity: {cosineSim(ref_ans,good_ans)}\nBM25B: {compare_bm25_binary(ref_score,good_score,80)}\nBM25: {compare_bm25(ref_score,good_score,80)}"
    newData.at[index,'good_clean_score'] = f"Entities: {compare_entities(clean_ref,clean_good)}\nSimilarity: {cosineSim(clean_ref,clean_good)}\nBM25B: {compare_bm25_binary(clean_ref_score,clean_good_score,80)}\nBM25: {compare_bm25(clean_ref_score,clean_good_score,80)}"
    newData.at[index,'good_noent_score'] = f"Entities: {compare_entities(noent_ref,noent_good)}\nSimilarity: {cosineSim(noent_ref,noent_good)}\nBM25B: {compare_bm25_binary(noent_ref_score,noent_good_score,80)}\nBM25: {compare_bm25(noent_ref_score,noent_good_score,80)}"

    newData.at[index,'fair_score'] = f"Entities: {compare_entities(ref_ans,fair_ans)}\nSimilarity: {cosineSim(ref_ans,fair_ans)}\nBM25B: {compare_bm25_binary(ref_score,fair_score,80)}\nBM25: {compare_bm25(ref_score,fair_score,80)}"
    newData.at[index,'fair_clean_score'] = f"Entities: {compare_entities(clean_ref,clean_fair)}\nSimilarity: {cosineSim(clean_ref,clean_fair)}\nBM25B: {compare_bm25_binary(clean_ref_score,clean_fair_score,80)}\nBM25: {compare_bm25(clean_ref_score,clean_fair_score,80)}"
    newData.at[index,'fair_noent_score'] = f"Entities: {compare_entities(noent_ref,noent_fair)}\nSimilarity: {cosineSim(noent_ref,noent_fair)}\nBM25B: {compare_bm25_binary(noent_ref_score,noent_fair_score,80)}\nBM25: {compare_bm25(noent_ref_score,noent_fair_score,80)}"

    newData.at[index,'partial_score'] = f"Entities: {compare_entities(ref_ans,partial_ans)}\nSimilarity: {cosineSim(ref_ans,partial_ans)}\nBM25B: {compare_bm25_binary(ref_score,partial_score,80)}\nBM25: {compare_bm25(ref_score,partial_score,80)}"
    newData.at[index,'partial_clean_score'] = f"Entities: {compare_entities(clean_ref,clean_partial)}\nSimilarity: {cosineSim(clean_ref,clean_partial)}\nBM25B: {compare_bm25_binary(clean_ref_score,clean_partial_score,80)}\nBM25: {compare_bm25(clean_ref_score,clean_partial_score,80)}"
    newData.at[index,'partial_noent_score'] = f"Entities: {compare_entities(noent_ref,noent_partial)}\nSimilarity: {cosineSim(noent_ref,noent_partial)}\nBM25B: {compare_bm25_binary(noent_ref_score,noent_partial_score,80)}\nBM25: {compare_bm25(noent_ref_score,noent_partial_score,80)}"

    newData.at[index,'fair_score'] = f"Entities: {compare_entities(ref_ans, fair_ans)}\nSimilarity: {cosineSim(ref_ans,fair_ans)}\nBM25B: {compare_bm25_binary(ref_score,fair_score,80)}\nBM25: {compare_bm25(ref_score,fair_score,80)}"
    newData.at[index,'fair_clean_score'] = f"Entities: {compare_entities(clean_ref,clean_fair)}\nSimilarity: {cosineSim(clean_ref,clean_fair)}\nBM25B: {compare_bm25_binary(clean_ref_score,clean_fair_score,80)}\nBM25: {compare_bm25(clean_ref_score,clean_fair_score,80)}"
    newData.at[index,'fair_noent_score'] = f"Entities: {compare_entities(noent_ref,noent_fair)}\nSimilarity: {cosineSim(noent_ref,noent_fair)}\nBM25B: {compare_bm25_binary(noent_ref_score,noent_fair_score,80)}\nBM25: {compare_bm25(noent_ref_score,noent_fair_score,80)}"

    newData.at[index,'wrong_score'] = f"Entities: {compare_entities(ref_ans,wrong_ans)}\nSimilarity: {cosineSim(ref_ans,wrong_ans)}\nBM25B: {compare_bm25_binary(ref_score,wrong_score,80)}\nBM25: {compare_bm25(ref_score,wrong_score,80)}"
    newData.at[index,'wrong_clean_score'] = f"Entities: {compare_entities(clean_ref,clean_wrong)}\nSimilarity: {cosineSim(clean_ref,clean_wrong)}\nBM25B: {compare_bm25_binary(clean_ref_score,clean_wrong_score,80)}\nBM25: {compare_bm25(clean_ref_score,clean_wrong_score,80)}"
    newData.at[index,'wrong_noent_score'] = f"Entities: {compare_entities(noent_ref,noent_wrong)}\nSimilarity: {cosineSim(noent_ref,noent_wrong)}\nBM25B: {compare_bm25_binary(noent_ref_score,noent_wrong_score,80)}\nBM25: {compare_bm25(noent_ref_score,noent_wrong_score,80)}"

    org_prf_list_entities.append(compare_entities(ref_ans, prf_ans))
    org_prf_list_similarities.append(cosineSim(ref_ans,prf_ans))
    org_prf_list_BM25B.append(compare_bm25_binary(ref_score,prf_score))
    org_prf_list_BM25.append(compare_bm25(ref_score,prf_score))

    clean_prf_list_entities.append(compare_entities(clean_ref, clean_prf))
    clean_prf_list_similarities.append(cosineSim(clean_ref,clean_prf))
    clean_prf_list_BM25B.append(compare_bm25_binary(clean_ref_score,clean_prf_score))
    clean_prf_list_BM25.append(compare_bm25(clean_ref_score,clean_prf_score))

    noent_prf_list_entities.append(compare_entities(noent_ref, noent_prf))
    noent_prf_list_similarities.append(cosineSim(noent_ref,noent_prf))
    noent_prf_list_BM25B.append(compare_bm25_binary(noent_ref_score,noent_prf_score))
    noent_prf_list_BM25.append(compare_bm25(noent_ref_score,noent_prf_score))

    org_good_list_entities.append(compare_entities(ref_ans, good_ans))
    org_good_list_similarities.append(cosineSim(ref_ans,good_ans))
    org_good_list_BM25B.append(compare_bm25_binary(ref_score,good_score))
    org_good_list_BM25.append(compare_bm25(ref_score,good_score))

    clean_good_list_entities.append(compare_entities(clean_ref, clean_good))
    clean_good_list_similarities.append(cosineSim(clean_ref,clean_good))
    clean_good_list_BM25B.append(compare_bm25_binary(clean_ref_score,clean_good_score))
    clean_good_list_BM25.append(compare_bm25(clean_ref_score,clean_good_score))

    if float(compare_entities(noent_ref, noent_good)) > float(compare_entities(noent_prf, noent_good)):
        noent_good_list_entities.append(compare_entities(noent_ref, noent_good))
    else:
        noent_good_list_entities.append(compare_entities(noent_prf, noent_good))


    noent_good_list_similarities.append(cosineSim(noent_ref,noent_good))
    noent_good_list_BM25B.append(compare_bm25_binary(noent_ref_score,noent_good_score))
    noent_good_list_BM25.append(compare_bm25(noent_ref_score,noent_good_score))

    org_wrong_list_entities.append(compare_entities(ref_ans, wrong_ans))
    org_wrong_list_similarities.append(cosineSim(ref_ans,wrong_ans))
    org_wrong_list_BM25B.append(compare_bm25_binary(ref_score,wrong_score))
    org_wrong_list_BM25.append(compare_bm25(ref_score,wrong_score))

    clean_wrong_list_entities.append(compare_entities(clean_ref, clean_wrong))
    clean_wrong_list_similarities.append(cosineSim(clean_ref,clean_wrong))
    clean_wrong_list_BM25B.append(compare_bm25_binary(clean_ref_score,clean_wrong_score))
    clean_wrong_list_BM25.append(compare_bm25(clean_ref_score,clean_wrong_score))

    if float(compare_entities(noent_ref, noent_wrong)) > float(compare_entities(noent_prf,noent_wrong)):
        noent_wrong_list_entities.append(compare_entities(noent_ref, noent_wrong))
    else:
        noent_wrong_list_entities.append(compare_entities(noent_prf, noent_wrong))



    noent_wrong_list_similarities.append(cosineSim(noent_ref,noent_wrong))
    noent_wrong_list_BM25B.append(compare_bm25_binary(noent_ref_score,noent_wrong_score))
    noent_wrong_list_BM25.append(compare_bm25(noent_ref_score,noent_wrong_score))

9_chunk_0.txt
14_chunk_0.txt
9_chunk_0.txt
14_chunk_0.txt
25_chunk_1.txt
8_chunk_2.txt
9_chunk_0.txt


/Users/alisa/anaconda3/lib/python3.11/site-packages/scipy/stats/_stats_py.py:5445: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  warnings.warn(stats.ConstantInputWarning(warn_msg))


14_chunk_0.txt
9_chunk_0.txt
14_chunk_0.txt
25_chunk_1.txt
8_chunk_2.txt
15_chunk_4.txt
4_chunk_0.txt
24_chunk_1.txt
17_chunk_0.txt
17_chunk_1.txt
3_chunk_0.txt
11_chunk_1.txt
7_chunk_1.txt
24_chunk_0.txt
20_chunk_3.txt
8_chunk_1.txt
16_chunk_0.txt
24_chunk_3.txt
10_chunk_0.txt
23_chunk_0.txt
5_chunk_0.txt
25_chunk_0.txt
19_chunk_2.txt
14_chunk_1.txt
16_chunk_1.txt
21_chunk_0.txt
15_chunk_1.txt
19_chunk_1.txt
24_chunk_2.txt
9_chunk_0.txt
14_chunk_0.txt
25_chunk_1.txt
8_chunk_2.txt
15_chunk_4.txt
4_chunk_0.txt
24_chunk_1.txt
17_chunk_0.txt
17_chunk_1.txt
3_chunk_0.txt
11_chunk_1.txt
7_chunk_1.txt
24_chunk_0.txt
20_chunk_3.txt
8_chunk_1.txt
16_chunk_0.txt
24_chunk_3.txt
10_chunk_0.txt
23_chunk_0.txt
5_chunk_0.txt
25_chunk_0.txt
19_chunk_2.txt
14_chunk_1.txt
16_chunk_1.txt
21_chunk_0.txt
15_chunk_1.txt
15_chunk_5.txt
19_chunk_1.txt
24_chunk_2.txt
20_chunk_0.txt
20_chunk_2.txt
18_chunk_1.txt
8_chunk_0.txt
20_chunk_1.txt
8_chunk_3.txt
18_chunk_2.txt
26_chunk_0.txt
15_chunk_0.txt
9_chunk_1.t

In [11]:
test=[]
for index,x in enumerate(zip(noent_good_list_entities,noent_wrong_list_entities)):
    z=x[1]
    x=x[0]
    test.append((index,float(x),float(z)))
print("mean", np.array(test).mean())
print("variance",np.array(test).var())


# Sort the list of tuples by the highest to lowest gap
sorted_data = sorted(test, key=sort_by_gap, reverse=False)
goodQuestions = sorted_data[:30]
indicesToKeep = [i[0] for i in goodQuestions]

mean 16.63019225213107
variance 796.168731094204


In [12]:
goodQuestions

[(50, 0.888888888888889, 0.0),
 (35, 0.8421052631578948, 0.0),
 (82, 0.7999999999999999, 0.0),
 (6, 0.8280254777070064, 0.11764705882352941),
 (8, 0.7906976744186047, 0.0),
 (26, 0.7759562841530053, 0.08000000000000002),
 (62, 0.7344632768361583, 0.023255813953488372),
 (25, 0.7323943661971831, 0.0),
 (90, 0.7368421052631579, 0.050632911392405056),
 (22, 0.7272727272727273, 0.0),
 (88, 0.7191011235955056, 0.0),
 (36, 0.72, 0.08),
 (52, 0.6986301369863014, 0.0),
 (14, 0.6993865030674846, 0.024096385542168672),
 (96, 0.7191011235955056, 0.12698412698412698),
 (64, 0.7037037037037038, 0.1),
 (84, 0.7407407407407406, 0.1875),
 (12, 0.75, 0.2),
 (71, 0.7659574468085107, 0.23076923076923073),
 (5, 0.6728971962616823, 0.058823529411764705),
 (79, 0.6666666666666667, 0.0),
 (21, 0.8380952380952381, 0.303030303030303),
 (29, 0.6896551724137931, 0.15),
 (20, 0.6528925619834711, 0.08108108108108107),
 (72, 0.64, 0.0),
 (34, 0.793103448275862, 0.2978723404255319),
 (54, 0.6728971962616822, 0.15686

In [13]:
test = []
for index, x in enumerate(zip(noent_good_list_similarities,noent_wrong_list_similarities)):
    if index in indicesToKeep:
        z=x[1]
        x=x[0]
        test.append((index,float(x),float(z)))

sorted_data = sorted(test, key=sort_by_gap, reverse=False)
bestQuestions = sorted_data

In [14]:
bestQuestions

[(82, 0.9067261815071106, 0.05423550307750702),
 (25, 0.9242154359817505, 0.13886010646820068),
 (62, 0.9018698930740356, 0.20087677240371704),
 (16, 0.9266661405563354, 0.21151646971702576),
 (72, 0.8148126602172852, 0.13326865434646606),
 (8, 0.7856669425964355, 0.10346118360757828),
 (35, 0.7639753222465515, 0.12775301933288574),
 (88, 0.7204099893569946, 0.11801600456237793),
 (71, 0.8128947019577026, 0.2459946721792221),
 (14, 0.8149549961090088, 0.27290230989456177),
 (79, 0.7291972637176514, 0.2049906849861145),
 (36, 0.7607160806655884, 0.2470909059047699),
 (6, 0.7281267046928406, 0.2410368174314499),
 (64, 0.7820826768875122, 0.31153252720832825),
 (15, 0.6702284216880798, 0.20793867111206055),
 (96, 0.801243782043457, 0.35638684034347534),
 (75, 0.7025333642959595, 0.29650184512138367),
 (84, 0.7788139581680298, 0.37632086873054504),
 (50, 0.5989246964454651, 0.26221367716789246),
 (5, 0.8489404916763306, 0.4644026756286621),
 (26, 0.5672924518585205, 0.30226439237594604),
 

In [91]:
ANS = set([word for ent in list(doc.ents) for word in ent.text.split()])
SRD = set([word for ent in list(docs.ents) for word in ent.text.split()])

In [92]:
ANS = ' . '.join(ANS)
SRD = ' . '.join(SRD)

In [93]:
ANS

'examination . Fundoscopic . Ankle . cognitive . arteries . hypertensive . Neurological . Palpation . Auscultation . peripheral . heart . emergencies . status . retinopathy . carotid'

In [94]:
SRD

'examination . Fundoscopic . cognitive . hypertensive . Neurological . emergencies . status . retinopathy'

In [84]:
cosineSim(ANS,SRD)

'0.8618925213813782'

In [90]:
compare_entities(REFANS,STDANS)

'0.6956521739130436'